In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [16]:
# %pip install "sliderule>=5.6,<6" icesat2db geopandas pyproj numpy pandas scipy scikit-learn matplotlib pyyaml
# Restart the kernel if upgrading SlideRule.

from pathlib import Path
import copy
import pandas as pd
import yaml
from IPython.display import display
from src.remote_parallel_blocks import load_sites, run_sites
import icesat2db as isdb

### Settings

In [27]:
PROCESSING_CONFIG_PATH = Path("../config.yaml")
SITES_PATH = Path("../../scratch/climate_site_selection/regional_site_selection/"\
                    "candidates_ce67aa33f7ea25a1/selection_aec51d75bae2d8e0/n09/selected_sites.csv")
START_TIME = "2018-10-01T00:00:00Z"
END_TIME = "2019-01-01T00:00:00Z"
LENGTH_M = 30                      # Or 15.
SITE_INDEX = [0]                    # Position in the selected-site table.
MAX_RESOURCES = 5                 # Overpasses for this initial run.
GRAPH_DETAIL = 1                  # Eg/Ev scatter and fit; 0 disables plots.
RESOURCES = None                  # Discover overpasses in the date window.
# For an Ontario footprint, optionally use the known test acquisition:
# RESOURCES = ["ATL03_20181024210256_04010102_007_01.h5"]
# Explicit resources bypass date discovery and must intersect the chosen site.


### Load configuration and Inspect Sites

In [28]:
with PROCESSING_CONFIG_PATH.open() as stream:
    cfg = yaml.safe_load(stream)
remote_cfg = copy.deepcopy(cfg)
remote_cfg["parallel_blocks"]["rebinned"] = LENGTH_M
remote_cfg["parallel_blocks"]["DW"] = False  # Use ATL08 landcover for this remote run.

selected = load_sites(SITES_PATH)
display(selected[["site_id", "west", "south", "east", "north"]])


,site_id,west,south,east,north
0,S01,75.4,55.7,82.2,59.4
1,S02,48.4,62.9,57.0,66.6
2,S03,-151.1,62.0,-142.8,65.7
3,S04,116.5,68.3,127.2,72.0
4,S05,-62.1,48.5,-56.3,52.2
5,S06,115.6,53.0,121.9,56.7
6,S07,-104.7,53.0,-98.4,56.7
7,S08,115.5,59.3,123.1,63.0
8,S09,-77.0,54.8,-70.3,58.5


### Run the chosen site

In [29]:
runs = run_sites(
    selected.iloc[SITE_INDEX], remote_cfg, START_TIME, END_TIME,
    resources=RESOURCES, max_resources=MAX_RESOURCES,
    graph_detail=GRAPH_DETAIL,
)
run = next(runs, None)
if run is not None:
    display(run.diagnostics)
    display(run.results.head())
    display(run.segments.head())
else:
    print("No overpasses found in this window.")


2026-10-23 16:29:25,809 - WARNING - Warning, this environment is using an outdated client (v5.6.3). The code will run but some functionality supported by the server (v5.7.0) may not be available.


S01: 5 overpasses
  ATL03_20181014223522_02490106_007_01.h5


2026-10-23 16:29:37,007 - INFO - Starting proxy for atl03x to process 1 resource(s) with 1 thread(s)
2026-10-23 16:29:37,150 - INFO - (dataframe.lua:138) <AppServer.1707> ATL03_20181014223522_02490106_007_01.h5/gt1l generated 157976 rows and 18 columns
2026-10-23 16:29:37,151 - INFO - Successfully completed processing resource [1 out of 1]: ATL03_20181014223522_02490106_007_01.h5
2026-10-23 16:29:37,152 - INFO - Successfuly received 157976 x 24 dataframe (eofs=1, dups=0, ptls=0)
2026-10-23 16:29:37,423 - INFO - Writing output file: /tmp/tmp3_yzoz4g
2026-10-23 16:29:38,896 - INFO - Closing output file: /tmp/tmp3_yzoz4g
2026-10-23 16:29:48,017 - INFO - Starting proxy for atl03x to process 1 resource(s) with 1 thread(s)
2026-10-23 16:29:49,927 - INFO - (dataframe.lua:138) <AppServer.1713> ATL03_20181014223522_02490106_007_01.h5/gt1r generated 639801 rows and 18 columns
2026-10-23 16:29:49,927 - INFO - Successfully completed processing resource [1 out of 1]: ATL03_20181014223522_02490106_0

,beam,status,photons,bins,matched_bins,accepted_bins
0,gt1l,atmosphere_rejected,157976,6765,901,0
1,gt1r,atmosphere_rejected,639801,9431,2156,0
2,gt2l,atmosphere_rejected,170158,7170,769,0
3,gt2r,atmosphere_rejected,404402,9357,1658,0
4,gt3l,atmosphere_rejected,118013,6893,714,0
5,gt3r,atmosphere_rejected,477708,9934,1718,0


,folder,date,lon,lat,pvpg,pv,pg,Eg,Ev,data_quantity,...,layer_flag,bin_id,atl08_segment_id,atl08_parent_count,atl08_parent_fraction,atl08_matched_photon_fraction,resource,site_id,length_m,canopy_openness_resolution_m


,resource,beam,spot,bin_id,length_m,coordinate_start,coordinate_end,longitude,latitude,n_te_photons,...,h_te_best_fit,terrain_slope,cloud_flag_atm,layer_flag,sat_flag,atl08_segment_id,atl08_segment_id_beg,used_for_processing,pipeline_beam,site_id
0,ATL03_20181014223522_02490106_007_01.h5,gt1l,6,449061,30,13471830,13471860,77.773464,59.221544,1,...,NaN,NaN,NaN,NaN,NaN,None,NaN,False,2,S01
1,ATL03_20181014223522_02490106_007_01.h5,gt1l,6,449062,30,13471860,13471890,77.773415,59.221301,3,...,NaN,NaN,NaN,NaN,NaN,None,NaN,False,2,S01
2,ATL03_20181014223522_02490106_007_01.h5,gt1l,6,449063,30,13471890,13471920,77.773354,59.220996,12,...,NaN,NaN,NaN,NaN,NaN,None,NaN,False,2,S01
3,ATL03_20181014223522_02490106_007_01.h5,gt1l,6,449064,30,13471920,13471950,77.773306,59.220752,15,...,NaN,NaN,NaN,NaN,NaN,None,NaN,False,2,S01
4,ATL03_20181014223522_02490106_007_01.h5,gt1l,6,449065,30,13471950,13471980,77.773250,59.220469,13,...,NaN,NaN,NaN,NaN,NaN,None,NaN,False,2,S01


# Inspect joined data

In [26]:
if run is not None and not run.segments.empty:
    display(run.segments[[
        "beam", "bin_id", "photon_rate_te", "photon_rate_can_nr",
        "msw_flag", "layer_flag", "segment_landcover", "canopy_openness",
        "canopy_openness_resolution_m", "atl08_matched",
        "atl08_parent_count", "atl08_parent_fraction", "used_for_processing",
    ]].head(20))


,beam,bin_id,photon_rate_te,photon_rate_can_nr,msw_flag,layer_flag,segment_landcover,canopy_openness,canopy_openness_resolution_m,atl08_matched,atl08_parent_count,atl08_parent_fraction,used_for_processing
0,gt1l,449061,1.000000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
1,gt1l,449062,1.000000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
2,gt1l,449063,1.200000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
3,gt1l,449064,1.071429,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
4,gt1l,449065,1.000000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
5,gt1l,449066,1.000000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
6,gt1l,449067,1.166667,0.083333,NaN,NaN,NaN,NaN,100,False,0,NaN,False
7,gt1l,449068,1.000000,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
8,gt1l,449069,1.111111,0.000000,NaN,NaN,NaN,NaN,100,False,0,NaN,False
9,gt1l,449070,0.615385,0.615385,NaN,NaN,NaN,NaN,100,False,0,NaN,False


# Optional: all sites and overpasses

In [ ]:
# result_tables = []
# for item in run_sites(selected, remote_cfg, START_TIME, END_TIME,
#                       max_resources=None, graph_detail=0):
#     display(item.diagnostics)
#     result_tables.append(item.results)
# results = pd.concat(result_tables, ignore_index=True) if result_tables else pd.DataFrame()
